# BNPL Governance Workshop - Part 0: Environment Setup

Run this notebook **first**, before `kredivo-01.ipynb` (Schema Registry / Data Contracts / CSFLE), `kredivo-02.ipynb` (Flink SQL stream processing), and `kredivo-03.ipynb` (Tableflow). It checks and prepares everything those three notebooks assume is already in place:

1. A working Python 3 + Jupyter setup inside VS Code / Cursor.
2. A Python virtual environment with the required packages installed.
3. A `.env` file with your Confluent Cloud credentials.
4. (Optional, for `kredivo-02.ipynb` only) the Confluent CLI installed and logged in, so the Flink REST calls actually execute instead of just printing SQL.
5. A live connectivity smoke test against your Confluent Cloud cluster.

Every check below prints ✅ / ⚠️ / ❌ so you can see at a glance what still needs fixing.

## 1. One-time IDE setup (do this manually, outside the notebook)

Before running any cells:

**VS Code**
1. Install the **Python** extension (`ms-python.python`).
2. Install the **Jupyter** extension (`ms-toolsai.jupyter`).
3. Open this folder (`kredivo/`) as your VS Code workspace.
4. Open this notebook, click the kernel picker in the top-right, and select the Python interpreter for `.venv` in this folder (create it in Step 2 below if it doesn't exist yet).

**Cursor**
1. Cursor bundles Python + Jupyter notebook support out of the box, but confirm both are enabled in Extensions (they're built-in / pre-installed by default — search "Python" and "Jupyter" to verify they're not disabled).
2. Open this folder as your Cursor workspace.
3. Open this notebook and select the `.venv` interpreter from the kernel picker, same as VS Code.

**Either editor**
- Make sure you have **Python 3.10+** installed system-wide (`python3 --version` in a terminal).
- You'll need a **Confluent Cloud** account with an environment, Kafka cluster, Schema Registry (Advanced Governance Package), and (for Part 2) a Flink compute pool already provisioned — ask your workshop host if you don't have one.
- The Governance Package of Schema Registry must be **Advanced** in order to use Data Contract and CSFLE.
- The cloud provider must be **AWS** in order to use TableFlow.


## 2. Check Python version and virtual environment

This repo already ships a `.venv/`. This cell just verifies you're actually running inside it (rather than some other interpreter) and that the version is new enough. If you selected the wrong kernel in Step 1, this is where you'll see it.

In [6]:
import sys, os

OK, FAIL, WARN, INFO = "\u2705", "\u274c", "\u26a0\ufe0f", "\u2139\ufe0f"

print(f"{INFO} Python executable : {sys.executable}")
print(f"{INFO} Python version    : {sys.version.split()[0]}")

major, minor = sys.version_info[:2]
if (major, minor) < (3, 10):
    print(f"{FAIL} Python 3.10+ required -- you have {major}.{minor}. Pick a newer interpreter from the kernel picker.")
else:
    print(f"{OK} Python version is new enough.")

in_venv = ".venv" in sys.executable or (hasattr(sys, "base_prefix") and sys.base_prefix != sys.prefix)
if in_venv:
    print(f"{OK} Running inside a virtual environment.")
else:
    print(f"{WARN} Doesn't look like you're in a virtual environment. In the notebook's kernel picker "
          f"(top-right), choose 'Select Another Kernel' -> 'Python Environments' -> the interpreter under "
          f"'{os.path.abspath('.venv')}' (create it first with the cell below if missing).")

ℹ️ Python executable : /Users/ahartono/Documents/workspace/workshops/kredivo/.venv/bin/python
ℹ️ Python version    : 3.14.6
✅ Python version is new enough.
✅ Running inside a virtual environment.


### If `.venv` doesn't exist yet

Run this from a terminal in this folder (not needed if `.venv/` is already present, as it is in this repo):

```bash
python3 -m venv .venv
source .venv/bin/activate   # Windows: .venv\Scripts\activate
```

Then reopen the kernel picker in VS Code / Cursor and select that interpreter.

## 3. Install dependencies

Installs everything both `kredivo-01.ipynb` and `kredivo-02.ipynb` import: the Kafka client, Schema Registry / JSON Schema / CSFLE support, the CEL rule executor, `.env` loading, and pandas for displaying results.

In [7]:
RUN_IN_NOTEBOOK = True  # Set to False if you'd rather copy PIP_CMD and run it yourself in a terminal (with .venv activated)

PIP_CMD = "pip install -q confluent-kafka python-dotenv jsonschema pandas requests cel-python"

if RUN_IN_NOTEBOOK:
    !{PIP_CMD}
else:
    print(f"Skipping -- run this yourself in a terminal (with .venv activated):\n\n  $ {PIP_CMD}")

## 4. Set up your `.env` file

All four workshop notebooks load credentials from a single `.env` file **in this same directory**. If it doesn't exist yet, copy the template and fill in your values:

```bash
cp .env.example .env
```

Then edit `.env` and fill in each value. The quick-reference table has the short version; the
walkthrough below it shows exactly where to click in Confluent Cloud Console for each one.

| Variable | Used by | Where to find it |
|---|---|---|
| `CCLOUD_BOOTSTRAP_SERVERS` | 01, 02 | Cloud Console -> cluster -> Cluster Settings |
| `CCLOUD_API_KEY` / `CCLOUD_API_SECRET` | 01, 02 | Cloud Console -> cluster -> API Keys |
| `SCHEMA_REGISTRY_URL` | 01, 02 | Cloud Console -> environment -> Schema Registry |
| `SCHEMA_REGISTRY_API_KEY` / `SCHEMA_REGISTRY_API_SECRET` | 01, 02 | Cloud Console -> environment -> Schema Registry -> API Keys |
| `FLINK_COMPUTE_POOL_ID` | 02 | Cloud Console -> environment -> Flink -> Compute Pools |
| `CONFLUENT_CLUSTER_ID` | 02, 03 | Cloud Console -> cluster -> Cluster Settings |
| `ORG_ID` | 02, 03 | `confluent organization list` |
| `ENV_ID` | 02, 03 | Cloud Console URL or `confluent environment list` |
| `CLOUD_REGION` / `CLOUD_PROVIDER` | 02, 03 | Cloud Console -> Flink -> Compute Pools (e.g. region `asia-southeast2`, provider `gcp`) |
| `FLINK_API_KEY` / `FLINK_API_SECRET` | 02 | A **separate** API key scoped to Flink -- Cloud Console -> environment -> Flink -> API Keys, or `confluent flink region list` + `confluent api-key create --resource <flink-region-id>` |
| `TABLEFLOW_API_KEY` / `TABLEFLOW_API_SECRET` | 03 | A **separate** API key scoped to Tableflow -- Cloud Console -> environment -> Tableflow -> API Keys, or `confluent api-key create --resource <environment-id>` |
| `PARTICIPANT_ID` | 01, 02, 03 | Optional -- defaults to your OS username; set this to namespace your topics if multiple people share a username |

### Step-by-step: getting every key, secret, and ID

All of this happens at [confluent.cloud](https://confluent.cloud), after logging in and selecting
the environment your workshop host gave you access to (top-left environment picker). Screens
occasionally get reorganized by Confluent, but the menu names below should still get you there.

#### a) Kafka cluster: `CCLOUD_BOOTSTRAP_SERVERS`, `CCLOUD_API_KEY`, `CCLOUD_API_SECRET`

1. From the environment page, click into your **Kafka cluster** (left sidebar, or the cluster
   tile on the environment overview page).
2. Click **Cluster Settings** in the left sidebar (sometimes called **Cluster Overview**).
3. Copy the **Bootstrap server** value shown there -> `CCLOUD_BOOTSTRAP_SERVERS`. It looks like
   `pkc-xxxxx.us-east-2.aws.confluent.cloud:9092`.
4. In the same cluster, click **API Keys** in the left sidebar -> **+ Add key**.
5. Choose **My account** (simplest for a workshop) or a service account if your host asks you to,
   then select **Global access** (or scope it down to this cluster only) -> **Create key**.
6. Confluent shows the **Key** and **Secret** exactly **once** -- copy both immediately into
   `CCLOUD_API_KEY` and `CCLOUD_API_SECRET`. If you miss it, delete the key and create a new one;
   there's no way to reveal a lost secret afterward.

#### b) Schema Registry: `SCHEMA_REGISTRY_URL`, `SCHEMA_REGISTRY_API_KEY`, `SCHEMA_REGISTRY_API_SECRET`

1. Go back to the **environment** level (not the cluster) -- Schema Registry is one per
   environment, shared by every cluster in it.
2. Click **Schema Registry** in the left sidebar. If it's not enabled yet, click **Enable** (your
   workshop host may have already done this).
3. Copy the **Endpoint URL** shown there -> `SCHEMA_REGISTRY_URL`. It looks like
   `https://psrc-xxxxx.us-east-2.aws.confluent.cloud`.
4. Still on the Schema Registry page, click **API Keys** -> **+ Add key** -> **My account** ->
   **Create key**.
5. Same one-time reveal as above -- copy **Key** and **Secret** immediately into
   `SCHEMA_REGISTRY_API_KEY` and `SCHEMA_REGISTRY_API_SECRET`.
6. Grant the key the right permissions:
   - Click **Schema Registry** in the left sidebar, then **API Keys**.
   - Find the key you just created, click the **⋮** (three dots) on the right, and choose
     **Show Details**.
   - In the API Key Detail panel, click **Associated account**, open the **Granted Permission**
     tab, and click **Grant Access**.
   - Add **DeveloperWrite**, **DeveloperRead**, and **EnvironmentAdmin**.

#### c) Cluster and environment IDs: `CONFLUENT_CLUSTER_ID`, `ENV_ID`, `ORG_ID`

These are identifiers, not secrets -- you can read them straight off the Console or the CLI.

- **`CONFLUENT_CLUSTER_ID`** -- on the cluster's **Cluster Settings** page (same page as 4a), look
  for **Cluster ID**. It looks like `lkc-xxxxxxx`.
- **`ENV_ID`** -- visible in the browser URL once you're inside an environment:
  `https://confluent.cloud/environments/env-xxxxxx/...` -- the `env-xxxxxx` part is your `ENV_ID`.
- **`ORG_ID`** -- click your account/organization name in the bottom-left corner ->
  **Organization Settings**, or run `confluent organization list` if you have the CLI (Step 5
  below) -- it's a UUID like `eb2976d2-949c-4e74-88ef-848321cca158`.

If you have the Confluent CLI installed and logged in (Step 5 below), all three of these are also
visible from the terminal:

```bash
confluent organization list      # -> ORG_ID
confluent environment list       # -> ENV_ID
confluent kafka cluster list     # -> CONFLUENT_CLUSTER_ID
```

#### d) Flink compute pool and region: `FLINK_COMPUTE_POOL_ID`, `CLOUD_REGION`, `CLOUD_PROVIDER`

1. From the environment page, click **Flink** in the left sidebar -> **Compute Pools**.
2. If you don't have one yet, click **+ Add Compute Pool**, pick a cloud provider/region, and
   create it (ask your workshop host if you're unsure which region to pick -- it must match the
   region of your Kafka cluster for Flink to read/write its topics).
3. Copy the pool's ID -> `FLINK_COMPUTE_POOL_ID`. It looks like `lfcp-xxxxxxx`.
4. The same Compute Pools page shows the pool's **Cloud** (e.g. `AWS`, `GCP`, `Azure`) and
   **Region** (e.g. `us-east-2`, `asia-southeast2`) -> `CLOUD_PROVIDER` and `CLOUD_REGION`
   (lowercase both, e.g. `aws` / `us-east-2`).

#### e) Flink API key: `FLINK_API_KEY`, `FLINK_API_SECRET`

This is a **separate** key from your Kafka key (4a) -- it authenticates to the Flink SQL REST API,
not the Kafka cluster.

1. From the environment page, click **Flink** -> **API Keys** in the left sidebar.
2. Click **+ Add API key** -> **My account** -> select this environment's Flink region -> **Create
   key**.
3. Copy **Key** and **Secret** immediately into `FLINK_API_KEY` and `FLINK_API_SECRET`.

Or via the CLI:

```bash
confluent flink region list
confluent api-key create --resource <flink-region-id>
```

#### f) Tableflow API key: `TABLEFLOW_API_KEY`, `TABLEFLOW_API_SECRET`

Only needed for `kredivo-03.ipynb`. Also a separate key, scoped to Tableflow's Iceberg REST
Catalog rather than Kafka or Flink.

1. From the environment page, click **Tableflow** in the left sidebar -> **API Keys**.
2. Click **+ Add API key** -> **My account** -> **Create key**.
3. Copy **Key** and **Secret** immediately into `TABLEFLOW_API_KEY` and `TABLEFLOW_API_SECRET`.

Or via the CLI:

```bash
confluent api-key create --resource <environment-id>
```

#### g) `PARTICIPANT_ID` (optional)

Not a Confluent Cloud value at all -- just a short, unique label (letters/numbers/`-`/`_`) that
namespaces every topic this workshop creates, so multiple attendees running the same notebooks
against a shared cluster don't collide. Leave it unset to default to your OS username.

The cell below just confirms the file exists and reports which variables are still missing -- it doesn't print any secret values.

In [8]:
import os
from dotenv import load_dotenv

OK, FAIL, WARN, INFO = "✅", "❌", "⚠️", "ℹ️"

ENV_PATH = ".env"

if not os.path.exists(ENV_PATH):
    print(f"{FAIL} No .env file found at {os.path.abspath(ENV_PATH)}.")
    print(f"     Run: cp .env.example .env   and fill in your values.")
else:
    load_dotenv(ENV_PATH)
    print(f"{OK} Found .env at {os.path.abspath(ENV_PATH)}")

    required_01_02 = [
        "CCLOUD_BOOTSTRAP_SERVERS", "CCLOUD_API_KEY", "CCLOUD_API_SECRET",
        "SCHEMA_REGISTRY_URL", "SCHEMA_REGISTRY_API_KEY", "SCHEMA_REGISTRY_API_SECRET",
    ]
    required_02_only = [
        "FLINK_COMPUTE_POOL_ID", "CONFLUENT_CLUSTER_ID",
        "ORG_ID", "ENV_ID", "CLOUD_REGION", "CLOUD_PROVIDER",
        "FLINK_API_KEY", "FLINK_API_SECRET",
    ]
    required_03_only = [
        "TABLEFLOW_API_KEY", "TABLEFLOW_API_SECRET",
    ]

    missing_01_02 = [v for v in required_01_02 if not os.getenv(v)]
    missing_02_only = [v for v in required_02_only if not os.getenv(v)]
    missing_03_only = [v for v in required_03_only if not os.getenv(v)]

    if missing_01_02:
        print(f"{FAIL} Missing (required for kredivo-01 and kredivo-02): {', '.join(missing_01_02)}")
    else:
        print(f"{OK} All Kafka + Schema Registry variables are set -- kredivo-01.ipynb is ready to run.")

    if missing_02_only:
        print(f"{WARN} Missing (only required for kredivo-02's Flink cells): {', '.join(missing_02_only)}")
        print(f"     Without these, kredivo-02.ipynb still runs, but Flink cells print SQL instead of executing it.")
    else:
        print(f"{OK} All Flink REST variables are set -- kredivo-02.ipynb's Flink cells will execute for real.")

    if missing_03_only:
        print(f"{WARN} Missing (only required for kredivo-03's Tableflow query cells): {', '.join(missing_03_only)}")
        print(f"     Without these, kredivo-03.ipynb still runs, but the pyiceberg/DuckDB query cells print what they'd run instead of executing it.")
    else:
        print(f"{OK} All Tableflow query variables are set -- kredivo-03.ipynb's query cells will execute for real.")

✅ Found .env at /Users/ahartono/Documents/workspace/workshops/kredivo/.env
✅ All Kafka + Schema Registry variables are set -- kredivo-01.ipynb is ready to run.
✅ All Flink REST variables are set -- kredivo-02.ipynb's Flink cells will execute for real.
✅ All Tableflow query variables are set -- kredivo-03.ipynb's query cells will execute for real.


## 5. (Optional, for kredivo-02.ipynb) Install and log in to the Confluent CLI

`kredivo-02.ipynb` talks to Flink over the REST API directly using the `FLINK_API_KEY` / `FLINK_API_SECRET` from `.env` -- it does **not** call the Confluent CLI itself. But you still need the CLI once, up front, to *obtain* those Flink-scoped credentials and IDs (`ORG_ID`, `ENV_ID`, `CLOUD_REGION`, `CLOUD_PROVIDER`, the Flink API key/secret) if you don't already have them.

Install:

```bash
# macOS
brew install confluentinc/tap/cli

# or see https://docs.confluent.io/confluent-cli/current/install.html for other platforms
```

Log in and fetch the IDs you need:

```bash
confluent login
confluent organization list
confluent environment list
confluent flink region list
confluent api-key create --resource <flink-region-id>
```

The cell below just checks whether the CLI is installed and whether you're currently logged in -- purely informational, nothing here is required for `kredivo-01.ipynb`.

In [ ]:
import subprocess

def run(cmd):
    try:
        return subprocess.run(cmd, capture_output=True, text=True, timeout=15)
    except FileNotFoundError:
        return None
    except Exception as e:
        print(f"{WARN} Error running {' '.join(cmd)}: {e}")
        return None

version_result = run(["confluent", "version"])
if version_result is None:
    print(f"{WARN} Confluent CLI not found on PATH. Only needed to look up Flink credentials/IDs once -- "
          f"install from https://docs.confluent.io/confluent-cli/current/install.html if you don't have your .env Flink values yet.")
else:
    print(f"{OK} Confluent CLI installed: {version_result.stdout.strip().splitlines()[0] if version_result.stdout else 'version unknown'}")
    who = run(["confluent", "api-key", "list", "--current-user"])
    # A quick, low-risk way to tell if we're logged in: any auth-requiring command works.
    ctx = run(["confluent", "context", "list"])
    if ctx and ctx.returncode == 0 and ctx.stdout.strip():
        print(f"{OK} A Confluent CLI context is configured (run `confluent context list` to see details).")
    else:
        print(f"{WARN} No active context found -- run `confluent login` if you still need to look up IDs/keys.")

✅ Confluent CLI installed: confluent - Confluent CLI


## 6. Connectivity smoke test

Confirms your `.env` credentials actually work end-to-end: connects to Schema Registry and lists subjects, then connects to the Kafka cluster's admin API. This is the same connection logic both workshop notebooks rely on -- if this cell passes, you're ready to open `kredivo-01.ipynb`.

In [ ]:
from confluent_kafka.admin import AdminClient
from confluent_kafka.schema_registry import SchemaRegistryClient

load_dotenv(ENV_PATH)

bootstrap_servers = os.getenv("CCLOUD_BOOTSTRAP_SERVERS")
kafka_api_key = os.getenv("CCLOUD_API_KEY")
kafka_api_secret = os.getenv("CCLOUD_API_SECRET")
sr_url = os.getenv("SCHEMA_REGISTRY_URL")
sr_api_key = os.getenv("SCHEMA_REGISTRY_API_KEY")
sr_api_secret = os.getenv("SCHEMA_REGISTRY_API_SECRET")

if not all([bootstrap_servers, kafka_api_key, kafka_api_secret, sr_url, sr_api_key, sr_api_secret]):
    print(f"{FAIL} Required Kafka / Schema Registry variables are missing from .env -- fix Step 4 above first.")
else:
    # Schema Registry check
    try:
        sr_client = SchemaRegistryClient({"url": sr_url, "basic.auth.user.info": f"{sr_api_key}:{sr_api_secret}"})
        subjects = sr_client.get_subjects()
        print(f"{OK} Connected to Schema Registry at {sr_url} -- {len(subjects)} existing subject(s).")
    except Exception as e:
        print(f"{FAIL} Could not reach Schema Registry: {e}")

    # Kafka admin check
    try:
        admin = AdminClient({
            "bootstrap.servers": bootstrap_servers,
            "security.protocol": "SASL_SSL",
            "sasl.mechanisms": "PLAIN",
            "sasl.username": kafka_api_key,
            "sasl.password": kafka_api_secret,
        })
        cluster_md = admin.list_topics(timeout=15)
        print(f"{OK} Connected to Kafka cluster at {bootstrap_servers} -- {len(cluster_md.topics)} existing topic(s) visible.")
    except Exception as e:
        print(f"{FAIL} Could not reach Kafka cluster: {e}")

    print(f"\n{INFO} If both checks above show {OK}, you're ready to open kredivo-01.ipynb.")
    print(f"{INFO} For kredivo-02.ipynb's Flink cells, also make sure Step 4's Flink variables are all set.")

## You're ready

If the checks above passed:
1. Open **`kredivo-01.ipynb`** -- Schema Registry, Data Contracts, and CSFLE.
2. Then open **`kredivo-02.ipynb`** -- Flink SQL stream processing (dedup, filter, enrich).
3. Then open **`kredivo-03.ipynb`** -- Tableflow (Iceberg tables on top of Part 2's enriched topic).

All three notebooks reuse the same `.env` file you set up here.